In [ ]:
from pathlib import Path
import json

from IPython.display import display, Markdown

from resources.imports import *
import torch

from resources.MLdata import DATA
from resources.MLmodels import MODEL
from resources.MLfunc import (
    postprocess_resolve_artifacts,
    postprocess_list_runs,
    postprocess_load_artifacts,
    postprocess_load_data,
    postprocess_available_evaluations,
    postprocess_attach_results,
    postprocess_build_diagnostics,
    postprocess_output_dir,
    postprocess_save_open_figures,
    print_field_diagnostics,
    plot_field_diagnostics,
    plot_field_sample,
)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


# ML Field Post-Processing

## 1. User Configuration

Paste a specific field-output run directory, model `.json`, model `.mdl`, regular HPO directory, or best-model HPO directory into `RUN_PATH`. If `RUN_PATH = None`, the notebook lists recent runs under `RUN_ROOT` and selects `RUN_INDEX`.

In [ ]:
RUN_ROOT = Path(r"Z:/p2")
RUN_PATH = None
RUN_INDEX = 0

PREFER_HPO_BEST = True
USE_SAVED_RESULTS_ONLY = True
LOAD_DATA = True
LOAD_MODEL = True
DEVICE = "cpu"

DATA_PATH_OVERRIDE = "auto"

RECOMPUTE_MISSING_DIAGNOSTICS_FROM_SAVED_PREDICTIONS = True
RECOMPUTE_MISSING_PREDICTIONS_WITH_MODEL = False
RUN_ACTIVATION_DIAGNOSTICS = False

SAVE_FIGURES = True
SAVE_RECOMPUTED_TABLES = True
OVERWRITE_POSTPROCESSING_OUTPUTS = True
POSTPROCESSING_LABEL = None

ACTIVE_MODE = None
ACTIVE_SPLIT = None

FIELD_FRAME = -1
FIELD_COMPONENT = 0
NODE_METRIC = "rmse"
RANDOM_SAMPLE_COUNT = 4
SELECTED_SAMPLE = 0


## 2. Find Recent Runs

In [ ]:
recent_runs = postprocess_list_runs(RUN_ROOT, max_runs=25, include_hpo=True)
display(recent_runs)

if RUN_PATH is None:
    if recent_runs.empty:
        raise ValueError("No saved runs were found. Set RUN_PATH manually.")
    RUN_PATH = Path(recent_runs.loc[int(RUN_INDEX), "run_dir"])

print("Selected RUN_PATH:", RUN_PATH)


## 3. Resolve And Load Saved Artifacts

Regular runs should resolve to `results/`. HPO best-model runs should resolve to `best_model_results/` when present.

In [ ]:
artifacts = postprocess_resolve_artifacts(
    RUN_PATH,
    run_root=RUN_ROOT,
    prefer_hpo_best=PREFER_HPO_BEST,
)

artifact_rows = []
for key, value in artifacts.items():
    if key == "hpo_candidate_model_jsons":
        artifact_rows.append((key, len(value)))
    elif key != "warnings":
        artifact_rows.append((key, str(value) if value is not None else None))
display(pd.DataFrame(artifact_rows, columns=["artifact", "path_or_value"]))

for warning in artifacts.get("warnings", []):
    print("WARNING:", warning)

loaded = postprocess_load_artifacts(artifacts)
POST_DIR = postprocess_output_dir(artifacts, label=POSTPROCESSING_LABEL, create=True)
print("Post-processing output directory:", POST_DIR)


## 4. Load DATA And MODEL

The data path is overridden to `RUN_ROOT` by default so HPC-side paths such as `/data/.../p2` can be reopened locally from `Z:/p2`. Field diagnostics can still be rebuilt from saved predictions if the DATA object is unavailable and the run summary contains the field shape.

In [ ]:
DAT = None
if LOAD_DATA and artifacts.get("data_json") is not None:
    try:
        DAT = postprocess_load_data(
            artifacts["data_json"],
            data_path_override=DATA_PATH_OVERRIDE,
            auto_path_root=RUN_ROOT,
        )
        print("Loaded DATA from:", artifacts["data_json"])
        print("DATA output_kind:", getattr(DAT, "output_kind", "curve"))
    except Exception as exc:
        print("DATA load failed:", repr(exc))

MOD = None
if LOAD_MODEL and artifacts.get("model_json") is not None and DAT is not None:
    try:
        MOD = MODEL.from_json(
            artifacts["model_json"],
            data=DAT,
            load_weights=artifacts.get("model_mdl") is not None,
            model_path=str(artifacts["model_mdl"]) if artifacts.get("model_mdl") is not None else None,
            device=DEVICE,
            scan_matches_on_init=False,
        )
        MOD = postprocess_attach_results(MOD, loaded)
        print("Loaded MODEL from:", artifacts["model_json"])
    except Exception as exc:
        print("MODEL load failed:", repr(exc))
elif LOAD_MODEL:
    print("MODEL was not loaded because a model JSON or DATA object is missing.")


## 5. Run Overview And HPO Files

In [ ]:
descriptor = loaded.get("descriptor") or {}
data_descriptor = loaded.get("data_descriptor") or {}
metrics = loaded.get("metrics") or {}
hpo = loaded.get("hpo") or {}

def _descriptor_output_kind(data_descriptor):
    if not isinstance(data_descriptor, dict):
        return None
    cfg = data_descriptor.get("data_config", data_descriptor.get("config", data_descriptor))
    return cfg.get("output_kind") if isinstance(cfg, dict) else None

saved_output_kind = _descriptor_output_kind(data_descriptor)
loaded_output_kind = getattr(DAT, "output_kind", None) if DAT is not None else saved_output_kind
print("Resolved output_kind:", loaded_output_kind)
if loaded_output_kind is not None and str(loaded_output_kind).lower() != "field":
    print("WARNING: this notebook is intended for field-output runs, but the selected run does not look like output_kind='field'.")

if descriptor:
    display(Markdown("### Model Descriptor"))
    display(pd.json_normalize(descriptor, sep=".").T.rename(columns={0: "value"}))
else:
    print("No model descriptor JSON was loaded.")

if data_descriptor:
    display(Markdown("### Data Descriptor"))
    display(pd.json_normalize(data_descriptor, sep=".").T.rename(columns={0: "value"}))

if hpo:
    display(Markdown("### HPO Artifacts"))
    for key, value in hpo.items():
        display(Markdown(f"#### {key}"))
        display(pd.json_normalize(value, sep=".").T.rename(columns={0: "value"}))
else:
    print("No HPO files were found for this run.")


## 6. Saved Metrics And Available Evaluations

In [ ]:
if metrics:
    display(Markdown("### Full metrics.json"))
    display(pd.json_normalize(metrics, sep=".").T.rename(columns={0: "value"}))

    scalar_metrics = {k: v for k, v in metrics.items() if isinstance(v, (str, int, float, bool)) or v is None}
    if scalar_metrics:
        display(Markdown("### Scalar Metrics"))
        display(pd.DataFrame(sorted(scalar_metrics.items()), columns=["metric", "value"]))
else:
    print("No metrics.json was found.")

available_evals = postprocess_available_evaluations(loaded)
display(Markdown("### Available Predictions / Diagnostic Tables"))
display(available_evals)

if ACTIVE_SPLIT is None:
    ACTIVE_SPLIT = metrics.get("evaluation_split", None)
if (ACTIVE_MODE is None or ACTIVE_SPLIT is None) and not available_evals.empty:
    if ACTIVE_MODE is None:
        ACTIVE_MODE = available_evals.iloc[0]["mode"]
    if ACTIVE_SPLIT is None:
        ACTIVE_SPLIT = available_evals.iloc[0]["split"]

print("ACTIVE_MODE:", ACTIVE_MODE)
print("ACTIVE_SPLIT:", ACTIVE_SPLIT)


## 7. Build In-Memory Field Diagnostics

This section may rebuild diagnostics from saved `predictions.npz` arrays. It does not run the neural network unless you explicitly enable the recompute-predictions section later.

In [ ]:
diagnostics = {}
recomputed_table_paths = []

for _, row in available_evals.iterrows():
    mode = str(row["mode"]).upper()
    split = str(row["split"]).lower()

    diag = postprocess_build_diagnostics(
        DAT,
        loaded,
        mode=mode,
        split=split,
        prefer_saved_tables=True,
        recompute_from_predictions=RECOMPUTE_MISSING_DIAGNOSTICS_FROM_SAVED_PREDICTIONS,
    )
    if diag is None:
        print(f"No prediction arrays available to build diagnostics for {mode} {split}.")
        continue
    if "field_shape" not in diag:
        print(f"Skipping {mode} {split}: diagnostics are not field diagnostics.")
        continue

    diagnostics[(mode, split)] = diag
    if MOD is not None:
        setattr(MOD, f"{mode}_{split}_diagnostics", diag)
        if split == "test":
            setattr(MOD, f"{mode}_diagnostics", diag)
            setattr(MOD, f"{mode}_prediction_summary", diag.get("summary"))

    print_field_diagnostics(diag, label=f"{mode} {split}")

    if SAVE_RECOMPUTED_TABLES:
        for table_key in ["sample_metrics", "frame_metrics", "component_metrics", "node_metrics"]:
            saved_key = f"{mode}_{split}_{table_key}"
            table = diag.get(table_key)
            was_saved = saved_key in loaded.get("diagnostic_tables", {})
            if hasattr(table, "to_csv") and not was_saved:
                out_path = POST_DIR / f"{saved_key}.csv"
                if OVERWRITE_POSTPROCESSING_OUTPUTS or not out_path.exists():
                    table.to_csv(out_path, index=True)
                    recomputed_table_paths.append(out_path)

if recomputed_table_paths:
    print("Saved recomputed diagnostic tables:")
    for path in recomputed_table_paths:
        print(" -", path)
elif SAVE_RECOMPUTED_TABLES:
    print("No diagnostic tables needed saving; existing run tables were reused or no diagnostics were built.")


## 8. Active Diagnostic Tables

In [ ]:
active_key = (str(ACTIVE_MODE).upper(), str(ACTIVE_SPLIT).lower()) if ACTIVE_MODE and ACTIVE_SPLIT else None
active_diag = diagnostics.get(active_key) if active_key is not None else None

if active_diag is None:
    print("No active diagnostics are available. Check predictions.npz, diagnostic CSVs, or ACTIVE_MODE/ACTIVE_SPLIT.")
else:
    display(Markdown(f"### Summary: {active_key[0]} {active_key[1]}"))
    display(pd.DataFrame(active_diag["summary"].items(), columns=["metric", "value"]))

    display(Markdown("### Frame Metrics"))
    display(active_diag.get("frame_metrics"))

    display(Markdown("### Component Metrics"))
    display(active_diag.get("component_metrics"))

    display(Markdown("### Sample Metrics"))
    display(active_diag.get("sample_metrics").head(20))

    display(Markdown("### Node Metrics"))
    node_metrics = active_diag.get("node_metrics")
    display(node_metrics.head(20) if hasattr(node_metrics, "head") else node_metrics)


## 9. Main Field Diagnostic Dashboard

In [ ]:
if active_diag is not None:
    plot_field_diagnostics(active_diag)


## 10. Frame And Component Error

In [ ]:
if active_diag is not None:
    frame = active_diag["frame_metrics"].copy()
    comp = active_diag["component_metrics"].copy()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes = axes.reshape(-1)

    axes[0].plot(frame["frame"], frame["rmse"], marker="o", label="RMSE")
    axes[0].plot(frame["frame"], frame["mae"], marker="o", label="MAE")
    axes[0].set_title("Frame Error")
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Error")
    axes[0].legend()

    axes[1].plot(frame["frame"], frame["bias"], color="black", marker="o")
    axes[1].axhline(0.0, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("Frame Bias")
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Prediction - Truth")

    axes[2].plot(frame["frame"], frame["valid_fraction"], color="tab:green", marker="o")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("Valid Fraction")
    axes[2].set_xlabel("Frame")
    axes[2].set_ylabel("Fraction")

    axes[3].bar(comp["component"].astype(str), comp["rmse"], color="tab:blue", alpha=0.85)
    axes[3].set_title("Component RMSE")
    axes[3].set_xlabel("Component")
    axes[3].set_ylabel("RMSE")

    fig.tight_layout()
    plt.show()


## 11. Prediction Collapse And Field Diversity

In [ ]:
if active_diag is not None:
    pred_std = active_diag.get("pred_std")
    true_std = active_diag.get("true_std")
    std_ratio = active_diag.get("std_ratio")

    if pred_std is None or true_std is None or std_ratio is None:
        print("Diversity arrays are unavailable for this diagnostics object.")
    else:
        pred_std = np.asarray(pred_std, dtype=float)
        true_std = np.asarray(true_std, dtype=float)
        std_ratio = np.asarray(std_ratio, dtype=float)
        frames = np.arange(pred_std.shape[0])
        frame_values = np.asarray(active_diag.get("frame_values", frames))
        x = frame_values if frame_values.shape[0] == pred_std.shape[0] else frames

        pred_frame_std = np.nanmean(pred_std, axis=(1, 2))
        true_frame_std = np.nanmean(true_std, axis=(1, 2))
        ratio_frame = np.nanmean(std_ratio, axis=(1, 2))

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(x, true_frame_std, label="Truth std", color="darkgreen")
        axes[0].plot(x, pred_frame_std, label="Prediction std", color="orangered")
        axes[0].set_title("Across-Sample Field Diversity")
        axes[0].set_xlabel("Frame")
        axes[0].set_ylabel("Mean std")
        axes[0].legend()

        axes[1].plot(x, ratio_frame, color="tab:purple")
        axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=1)
        axes[1].set_title("Prediction Std / Truth Std")
        axes[1].set_xlabel("Frame")
        axes[1].set_ylabel("Ratio")

        values = std_ratio[np.isfinite(std_ratio)]
        axes[2].hist(values, bins=40, color="tab:blue", alpha=0.8)
        axes[2].axvline(1.0, color="gray", linestyle="--", linewidth=1)
        axes[2].set_title("Node-Frame-Component Ratio")
        axes[2].set_xlabel("Std ratio")
        axes[2].set_ylabel("Count")

        fig.tight_layout()
        plt.show()


## 12. Frame-Component Heatmaps

In [ ]:
if active_diag is not None:
    y_pred = np.asarray(active_diag["y_pred"], dtype=float)
    y_true = np.asarray(active_diag["y_true"], dtype=float)
    valid = np.asarray(active_diag.get("valid_mask", np.isfinite(y_true) & np.isfinite(y_pred)), dtype=bool)
    err = y_pred - y_true

    n_frames = y_pred.shape[1]
    n_components = y_pred.shape[3]
    components = [str(c) for c in active_diag.get("components", [f"c{i}" for i in range(n_components)])]
    frame_values = np.asarray(active_diag.get("frame_values", np.arange(n_frames)))
    frame_labels = [f"{v:g}" if isinstance(v, (int, float, np.integer, np.floating)) else str(v) for v in frame_values]

    rmse_map = np.full((n_frames, n_components), np.nan)
    bias_map = np.full((n_frames, n_components), np.nan)
    valid_map = np.full((n_frames, n_components), np.nan)
    for frame_idx in range(n_frames):
        for comp_idx in range(n_components):
            mask = valid[:, frame_idx, :, comp_idx]
            values = err[:, frame_idx, :, comp_idx]
            if np.any(mask):
                rmse_map[frame_idx, comp_idx] = np.sqrt(np.nanmean(values[mask] ** 2))
                bias_map[frame_idx, comp_idx] = np.nanmean(values[mask])
                valid_map[frame_idx, comp_idx] = np.mean(mask)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, matrix, title, cmap in [
        (axes[0], rmse_map, "RMSE", "viridis"),
        (axes[1], bias_map, "Bias", "coolwarm"),
        (axes[2], valid_map, "Valid Fraction", "magma"),
    ]:
        im = ax.imshow(matrix, aspect="auto", cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("Component")
        ax.set_ylabel("Frame")
        ax.set_xticks(np.arange(n_components))
        ax.set_xticklabels(components)
        ax.set_yticks(np.arange(n_frames))
        ax.set_yticklabels(frame_labels)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


## 13. Node-Level Spatial Error

In [ ]:
if active_diag is not None:
    node = active_diag.get("node_metrics")
    if node is None or not hasattr(node, "copy"):
        print("Node metrics are unavailable.")
    else:
        node = node.copy()
        coords = active_diag.get("node_coords")
        if ("x" not in node.columns or "y" not in node.columns) and coords is not None and len(coords) == len(node):
            coords = np.asarray(coords, dtype=float)
            node["x"] = coords[:, 0]
            node["y"] = coords[:, 1]

        metric = NODE_METRIC if NODE_METRIC in node.columns else "rmse"
        display(Markdown(f"### Worst Nodes By {metric}"))
        display(node.sort_values(metric, ascending=False).head(20))

        if "x" in node.columns and "y" in node.columns:
            plot_cols = [col for col in ["rmse", "mae", "bias", "valid_fraction"] if col in node.columns]
            fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 4))
            axes = np.asarray(axes).reshape(-1)
            for ax, col in zip(axes, plot_cols):
                sc = ax.scatter(node["x"], node["y"], c=node[col], cmap="viridis", s=22)
                ax.set_aspect("equal", adjustable="box")
                ax.set_title(f"Node {col}")
                fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            plt.show()
        else:
            print("Node coordinates are unavailable, so spatial node maps cannot be drawn.")


## 14. Sample-Level Metric Distributions

In [ ]:
if active_diag is not None:
    samples = active_diag["sample_metrics"]
    numeric_cols = [
        "sample_mae",
        "sample_mse",
        "sample_rmse",
        "sample_bias",
        "valid_fraction",
    ]
    numeric_cols = [col for col in numeric_cols if col in samples.columns]

    ncols = 3
    nrows = int(np.ceil(len(numeric_cols) / ncols)) if numeric_cols else 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for ax, col in zip(axes, numeric_cols):
        ax.hist(samples[col].dropna(), bins=40, color="tab:blue", alpha=0.8)
        ax.set_title(col)
        ax.set_ylabel("Count")
    for ax in axes[len(numeric_cols):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()

    if numeric_cols:
        display(samples[numeric_cols].describe().T)


## 15. Best, Worst, Selected, And Random Fields

In [ ]:
if active_diag is not None:
    samples = active_diag["sample_metrics"]
    coords = active_diag.get("node_coords")
    if coords is None:
        print("Node coordinates are unavailable, so field sample maps cannot be drawn.")
    else:
        candidate_indices = []
        if "sample_rmse" in samples.columns:
            candidate_indices.extend(samples.sort_values("sample_rmse").head(1)["sample"].astype(int).tolist())
            candidate_indices.extend(samples.sort_values("sample_rmse").tail(1)["sample"].astype(int).tolist())
        candidate_indices.append(int(SELECTED_SAMPLE))
        rng = np.random.default_rng(42)
        if len(samples) > 0:
            candidate_indices.extend(rng.choice(samples["sample"].astype(int), size=min(RANDOM_SAMPLE_COUNT, len(samples)), replace=False).tolist())
        n_samples = active_diag["y_pred"].shape[0]
        candidate_indices = [idx for idx in dict.fromkeys(candidate_indices) if 0 <= idx < n_samples]

        for idx in candidate_indices:
            row = samples.loc[samples["sample"].astype(int) == int(idx)]
            rmse_text = f", RMSE={row['sample_rmse'].iloc[0]:.4g}" if not row.empty and "sample_rmse" in row else ""
            print(f"Sample {idx}{rmse_text}, frame={FIELD_FRAME}, component={FIELD_COMPONENT}")
            plot_field_sample(
                active_diag,
                sample=idx,
                frame=FIELD_FRAME,
                component=FIELD_COMPONENT,
                node_coords=coords,
            )


## 16. Selected Sample Frame Evolution

In [ ]:
if active_diag is not None:
    y_pred = np.asarray(active_diag["y_pred"], dtype=float)
    y_true = np.asarray(active_diag["y_true"], dtype=float)
    valid = np.asarray(active_diag.get("valid_mask", np.isfinite(y_true) & np.isfinite(y_pred)), dtype=bool)
    n_samples, n_frames, _, n_components = y_pred.shape
    sample_idx = int(np.clip(SELECTED_SAMPLE, 0, n_samples - 1))
    frame_values = np.asarray(active_diag.get("frame_values", np.arange(n_frames)))
    x = frame_values if frame_values.shape[0] == n_frames else np.arange(n_frames)
    components = [str(c) for c in active_diag.get("components", [f"c{i}" for i in range(n_components)])]

    pred_s = np.where(valid[sample_idx], y_pred[sample_idx], np.nan)
    true_s = np.where(valid[sample_idx], y_true[sample_idx], np.nan)
    err_s = pred_s - true_s

    pred_mean = np.nanmean(pred_s, axis=1)
    true_mean = np.nanmean(true_s, axis=1)
    mae_frame = np.nanmean(np.abs(err_s), axis=1)

    fig, axes = plt.subplots(1, n_components, figsize=(6 * n_components, 4), squeeze=False)
    for comp_idx in range(n_components):
        ax = axes[0, comp_idx]
        ax.plot(x, true_mean[:, comp_idx], label="Truth", color="darkgreen")
        ax.plot(x, pred_mean[:, comp_idx], label="Prediction", color="orangered")
        ax.plot(x, mae_frame[:, comp_idx], label="Mean abs error", color="gray", linestyle="--")
        ax.set_title(f"Sample {sample_idx} - {components[comp_idx]}")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Mean node value")
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


## 17. Loss Evaluation

For field runs this normally evaluates `MaskedFieldMSELoss` on the saved predictions and truth arrays. This is a diagnostic calculation only; it does not train or update the model.

In [ ]:
loss_rows = []
if MOD is not None and active_diag is not None:
    mode = active_key[0]
    y_pred_t = torch.as_tensor(active_diag["y_pred"], dtype=torch.float32, device=MOD.device)
    y_true_t = torch.as_tensor(active_diag["y_true"], dtype=torch.float32, device=MOD.device)
    mode_losses = list(getattr(MOD, f"{mode}_losses", []))
    if not mode_losses and hasattr(MOD, "losses"):
        mode_losses = list(getattr(MOD, "losses", []))

    for loss_idx, loss_obj in enumerate(mode_losses):
        try:
            with torch.no_grad():
                value = loss_obj(y_pred_t, y_true_t)
            value = value.detach().cpu().item() if torch.is_tensor(value) else float(value)
            loss_rows.append({"loss_index": loss_idx, "loss_class": loss_obj.__class__.__name__, "value": value})
        except Exception as exc:
            loss_rows.append({"loss_index": loss_idx, "loss_class": loss_obj.__class__.__name__, "value": np.nan, "note": repr(exc)})

loss_components_df = pd.DataFrame(loss_rows)
display(loss_components_df)

if not loss_components_df.empty and "value" in loss_components_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = loss_components_df["loss_class"] + " #" + loss_components_df["loss_index"].astype(str)
    ax.bar(labels, loss_components_df["value"], color="tab:blue")
    ax.set_title("Saved Prediction Loss")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    plt.show()
elif MOD is None:
    print("MODEL is not loaded, so loss values cannot be evaluated.")


## 18. Optional Activation Diagnostics

This requires a forward pass through the loaded model and dataloader. It is off by default.

In [ ]:
if RUN_ACTIVATION_DIAGNOSTICS:
    if MOD is None or ACTIVE_MODE is None or ACTIVE_SPLIT is None:
        print("Activation diagnostics require a loaded MODEL and active mode/split.")
    else:
        activation_summary = MOD.activation_diagnostics(
            mode=ACTIVE_MODE,
            split=ACTIVE_SPLIT,
            max_batches=1,
            plot=True,
        )
        display(activation_summary)
else:
    print("Activation diagnostics skipped. Set RUN_ACTIVATION_DIAGNOSTICS = True to enable.")


## 19. Optional Prediction Recompute

This is intentionally disabled by default. Enable only when saved prediction arrays are missing and you want to run the loaded model locally.

In [ ]:
if RECOMPUTE_MISSING_PREDICTIONS_WITH_MODEL:
    if USE_SAVED_RESULTS_ONLY:
        print("Skipped because USE_SAVED_RESULTS_ONLY = True.")
    elif MOD is None:
        print("Skipped because MODEL is not loaded.")
    elif ACTIVE_SPLIT is None:
        print("Skipped because ACTIVE_SPLIT is not set.")
    else:
        recomputed = MOD.evaluate_split(
            split=ACTIVE_SPLIT,
            mode=ACTIVE_MODE,
            plot=False,
            diagnostics=True,
            diag_plot=True,
        )
        print("Recomputed predictions in memory. Existing results files were not overwritten.")
else:
    print("Prediction recompute skipped.")


## 20. Save Figures

In [ ]:
if SAVE_FIGURES:
    prefix = ""
    if ACTIVE_MODE and ACTIVE_SPLIT:
        prefix = f"{str(ACTIVE_MODE).upper()}_{str(ACTIVE_SPLIT).lower()}"
    saved_figures = postprocess_save_open_figures(
        POST_DIR,
        prefix=prefix,
        formats=("png",),
        dpi=250,
        close=False,
    )
    print(f"Saved {len(saved_figures)} figure file(s) to {POST_DIR}")
    for path in saved_figures:
        print(" -", path)
else:
    print("Figure saving skipped.")
